```yaml
version: 1.0
author: Rolland Palffy (palffy.rolland@codespring.ro)
reviewers:
    - Laszlo Barabasi (Laszlo Barabas (barabas.laszlo@codespring.ro)
```

# Classes, Types, and Decorators

You're probably familiar with classes from other programming languages. In Python, the constructor is called `__init__()` and the convention is to use `self` instead of `this`, which is an explicit argument. Here's the definition of a class called `Vector2D`, which has `x` and `y` attributes and several methods:

In [1]:
class Vector2D:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def to_string(self):
        return f"Vector2D(x={self.x}, y={self.y})"

    def equal(self, other):
        return (self.x, self.y) == (other.x, other.y)

    def add(self, other):
        return Vector2D(self.x + other.x, self.y + other.y)

v1 = Vector2D(3, 4)
v2 = Vector2D(1, -2)
print(v1)
print(v1.to_string())
print(v1.equal(v2))
print(v1.add(v2).to_string())

Vector2D(x=3, y=4)
False
Vector2D(x=4, y=2)


## Inheritance

The syntax for a derived class definition looks like this:

In [2]:
class Vector3D(Vector2D):
    def __init__(self, x, y, z):
        super().__init__(x, y)
        self.z = z
    
    def __repr__(self):
        return f"Vector3D(x={self.x}, y={self.y}, z={self.z})"

print(Vector3D(1, 2, 3))

Vector3D(x=1, y=2, z=3)


Python supports multiple inheritance. What problems can it cause? How does `super()` work in that case?

## Private variables

Private instance variables don't exist in Python. However, there is a convention that is followed by most Python code: a name prefixed with an underscore (e.g. `_spam`) should be treated as a non-public part of the API.

There is limited support for class-private members, called *name mangling*. Any identifier of the form `__spam` (at least two leading underscores, at most one trailing underscore) is textually replaced with `_classname__spam`.

In [3]:
class Mapping:
    def __init__(self, iterable):
        self.items_list = []
        self.__update(iterable)

    def _update(self, iterable):
        for item in iterable:
            self.items_list.append(item)

    __update = _update

mapping = Mapping([1, 2, 3])
print(mapping.items_list)

mapping._update([4, 5, 6])
print(mapping.items_list)

mapping._Mapping__update([7, 8, 9])
print(mapping.items_list)

[1, 2, 3]
[1, 2, 3, 4, 5, 6]
[1, 2, 3, 4, 5, 6, 7, 8, 9]



## Dunder methods

*Dunder methods* (short for *double underscore methods*), also known as *magic methods* or special methods, are predefined methods in Python that start and end with double underscores (e.g., `__init__`). They enable custom classes to implement and interact with built-in Python behavior, such as object construction, string representation, arithmetic operations, comparisons, and more. For example, `__repr__` defines string representation, and `__add__` allows use of the `+` operator.

In [4]:
class Vector2D:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector2D(x={self.x}, y={self.y})"

    def __eq__(self, other):
        return isinstance(other, Vector2D) and (self.x, self.y) == (other.x, other.y)

    def __add__(self, other):
        if not isinstance(other, Vector2D):
            return NotImplemented
        return Vector2D(self.x + other.x, self.y + other.y)

v1 = Vector2D(3, 4)
v2 = Vector2D(1, -2)
print(v1)
print(v1 == v2)
print(v1 + v2)
try:
    print(v2 + 1)
except TypeError as e:
    print('TypeError:', e)

Vector2D(x=3, y=4)
False
Vector2D(x=4, y=2)
TypeError: unsupported operand type(s) for +: 'Vector2D' and 'int'


There is also `__str__` for string representation. How is it different from `__repr__`?

## Type hints

Type hints provide a standard syntax for type annotations, opening up Python code to easier static analysis and refactoring, potential runtime type checking (see: [Pydantic](https://docs.pydantic.dev/latest/)), and code generation utilizing type information.

**The runtime does not enforce function and variable type annotations**, but they can be used by third party tools. Enable type checking in VS Code to display error: [python.analysis.typeCheckingMode](vscode://settings/python.analysis.typeCheckingMode)

In [ ]:
class Vector2D:
    def __init__(self, x: int | float = 0, y: int | float = 0) -> None:
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        return f"Vector2D(x={self.x}, y={self.y})"

    def __eq__(self, other: "Vector2D") -> bool:
        return (self.x, self.y) == (other.x, other.y)

    def __add__(self, other: "Vector2D") -> "Vector2D": # or typing.Self
        return Vector2D(self.x + other.x, self.y + other.y)

print(Vector2D(2, 3) + 1) # AttributeError: 'int' object has no attribute 'x'

AttributeError: 'int' object has no attribute 'x'

Types are ordinary objects that can be accessed at runtime:

In [6]:
NumPair = tuple[int | float, int | float]
print(NumPair)

tuple[int | float, int | float]


## Decorators

Decorators add new functionality to classes, methods, and functions without modifying their structure.

In [7]:
from dataclasses import dataclass

@dataclass(frozen=True, order=True)
class Vector2D:
    x: int | float
    y: int | float

    @property
    def pair(self) -> tuple[int | float, int | float]:
        return self.x, self.y

    def __add__(self, other: "Vector2D") -> "Vector2D": # or typing.Self
        return Vector2D(self.x + other.x, self.y + other.y)
    
    @staticmethod
    def zero() -> "Vector2D":
        return Vector2D(0, 0)

v1 = Vector2D(2, 7)
v2 = Vector2D(3, 5)

# automatically implements string representation and all comparisons
print(v1)
print(v1 == v2, v1 < v2, v1 > v2)

print(v1.pair) # property
print(Vector2D.zero()) # static method call

Vector2D(x=2, y=7)
False True False
(2, 7)
Vector2D(x=0, y=0)


Decorators also support functions:

In [8]:
from functools import cache

@cache # try to comment it out and see difference in execution time
def fib(n):
    """Compute the nth Fibonacci number recursively."""
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2) # O(2^n)

fib(38)

39088169

### Custom decorator

A decorator is a higher-order function that **takes a function as an argument and returns a function**. Decorators can execute code when a decorated function is defined, when it is called, and when it returns, **observing all inputs and outputs**.

In [9]:
def log(func):
    """Logging decorator that prints function calls and results."""
    print(f"LOG: defined {func}")
    def wrapper(*args, **kwargs):
        print(f"LOG: called with {args=} {kwargs=}")
        result = func(*args, **kwargs)
        print(f"LOG: returned {result!r}")
        return result
    return wrapper

@log
def full_name(first_name, last_name, *, sep=" "): # sep is a keyword-only arg
    """Return the full name by joining first and last name with a separator."""
    return f"{first_name}{sep}{last_name}"

print("-" * 79)
full_name("John", "Doe", sep="_")

LOG: defined <function full_name at 0x1126c6200>
-------------------------------------------------------------------------------
LOG: called with args=('John', 'Doe') kwargs={'sep': '_'}
LOG: returned 'John_Doe'


'John_Doe'

Using a decorator with `@` is equivalent to *calling* the decorator with the function as an argument: `fn = log(full_name)`

To create a decorator *with arguments*, we need to create a *decorator factory* that wraps and returns the decorator itself:

In [10]:
def log_factory(*, prefix="LOG"):
    """A decorator factory that creates a logging decorator with a custom prefix."""
    def decorator(func):
        print(f"{prefix}: defined with {prefix=} {func}")
        def wrapper(*args, **kwargs):
            print(f"{prefix}: called with {args=} {kwargs=}")
            result = func(*args, **kwargs)
            print(f"{prefix}: returned {result!r}")
            return result
        return wrapper
    return decorator

@log_factory(prefix="CUSTOM_LOG")
def sum3(x, y, z, /): # positional only arguments
    """Return the sum of three numbers."""
    return x + y + z

print("-" * 79)
sum3(1, 2, 3)

CUSTOM_LOG: defined with prefix='CUSTOM_LOG' <function sum3 at 0x112712020>
-------------------------------------------------------------------------------
CUSTOM_LOG: called with args=(1, 2, 3) kwargs={}
CUSTOM_LOG: returned 6


6